# 7.4 安全对齐 (Safety Alignment)

安全对齐确保模型不会生成有害、偏见或危险的内容，是模型部署前的关键步骤。没有安全对齐的模型可能被恶意利用，造成严重后果。

## 安全对齐的层次

1. **预防层**：红队测试发现漏洞，安全微调修补漏洞
2. **检测层**：输入/输出过滤，实时检测有害内容
3. **响应层**：拒绝有害请求，提供安全替代回复
4. **持续改进层**：宪法AI自我修正，持续监控和更新

## 安全对齐的挑战
- **过度安全（Over-refusal）**：拒绝本应回答的正常问题
- **安全与有用性的权衡**：过于保守会降低模型有用性
- **对抗性攻击**：精心构造的输入可以绕过安全措施
- **文化差异**：不同文化对"安全"的定义不同

## 1. 红队测试（Red Teaming）

红队测试是主动发现模型安全漏洞的方法，通过系统性地构造对抗性输入来测试模型的安全性。

### 红队测试方法
- **人工红队**：安全专家手动构造攻击提示
- **自动红队**：使用另一个模型自动生成攻击提示
- **梯度引导红队**：基于梯度的方法寻找最可能触发有害输出的输入
- **变异红队**：对已知攻击进行变异生成新攻击

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SafetyClassifier(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 128), nn.ReLU(),
            nn.LayerNorm(128),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.net(x)

class RedTeamer:
    def __init__(self, target_model, safety_classifier, d=64):
        self.target = target_model
        self.classifier = safety_classifier
        self.d = d

    def random_attack(self, n_attacks=100):
        attacks = torch.randn(n_attacks, self.d)
        with torch.no_grad():
            safety_logits = self.classifier(attacks)
            unsafe_mask = safety_logits.argmax(dim=-1) == 1
        successful = attacks[unsafe_mask]
        return successful, unsafe_mask.float().mean().item()

    def gradient_guided_attack(self, n_steps=50, lr=0.1, target_class=1):
        x = torch.randn(1, self.d, requires_grad=True)
        for step in range(n_steps):
            logits = self.classifier(x)
            target_logit = logits[0, target_class]
            grad = torch.autograd.grad(target_logit, x)[0]
            x = x + lr * grad.sign()
            x = x.detach().requires_grad_(True)
        with torch.no_grad():
            final_logits = self.classifier(x)
            success = final_logits.argmax(dim=-1).item() == target_class
        return x.detach(), success

    def mutation_attack(self, seed_attack, n_mutations=20, noise_scale=0.3):
        mutations = seed_attack.unsqueeze(0) + noise_scale * torch.randn(n_mutations, self.d)
        with torch.no_grad():
            safety_logits = self.classifier(mutations)
            unsafe_mask = safety_logits.argmax(dim=-1) == 1
        successful = mutations[unsafe_mask]
        return successful, unsafe_mask.float().mean().item()

classifier = SafetyClassifier()
target = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 50))

red_team = RedTeamer(target, classifier, d=64)

print('=== Red Teaming ===')

successful, rate = red_team.random_attack(n_attacks=200)
print(f'\nRandom attack: {successful.shape[0]}/200 successful ({rate:.1%})')

adv_input, success = red_team.gradient_guided_attack(n_steps=50, lr=0.1)
print(f'Gradient-guided attack: success={success}')

if successful.shape[0] > 0:
    mutated, mut_rate = red_team.mutation_attack(successful[0], n_mutations=50)
    print(f'Mutation attack: {mutated.shape[0]}/50 successful ({mut_rate:.1%})')

print(f'\nKey: Gradient-guided attacks are most effective for finding vulnerabilities.')
print(f'Mutation attacks explore nearby adversarial space efficiently.')

## 2. 安全微调

安全微调通过在安全数据上训练模型，使其学会拒绝有害请求、提供安全回复。

### 安全微调数据构建
- **有害请求 + 拒绝回复**：教会模型拒绝
- **边界请求 + 谨慎回复**：处理灰色地带
- **正常请求 + 正常回复**：防止过度安全
- **越狱尝试 + 坚定拒绝**：增强抗攻击能力

### 过度安全问题
过度安全（Over-refusal）是安全微调最常见的副作用：
- "如何烹饪鸡肉？" → "我无法帮助您烹饪。"（错误拒绝）
- "如何制作武器？" → "我无法帮助您。"（正确拒绝）

In [ ]:
class SafetyFineTuner:
    def __init__(self, model, d=64, n_classes=50):
        self.model = model
        self.d = d
        self.n_classes = n_classes
        self.safety_categories = {
            'violence': {'weight': 1.0, 'n_examples': 500},
            'hate_speech': {'weight': 1.0, 'n_examples': 500},
            'self_harm': {'weight': 1.0, 'n_examples': 300},
            'sexual': {'weight': 0.8, 'n_examples': 400},
            'misinformation': {'weight': 0.7, 'n_examples': 300},
            'borderline': {'weight': 0.5, 'n_examples': 200},
            'normal_safe': {'weight': 0.3, 'n_examples': 500},
        }

    def generate_safety_data(self, n_per_category=32):
        data = []
        for cat, info in self.safety_categories.items():
            x = torch.randn(n_per_category, self.d)
            if cat == 'normal_safe':
                labels = torch.randint(0, self.n_classes, (n_per_category,))
                is_safe = torch.ones(n_per_category, dtype=torch.bool)
            elif cat == 'borderline':
                labels = torch.randint(0, self.n_classes, (n_per_category,))
                is_safe = torch.ones(n_per_category, dtype=torch.bool)
            else:
                reject_token = 0
                labels = torch.full((n_per_category,), reject_token, dtype=torch.long)
                is_safe = torch.zeros(n_per_category, dtype=torch.bool)
            weight = torch.full((n_per_category,), info['weight'])
            data.append((x, labels, is_safe, weight, cat))
        return data

    def train_step(self, data, optimizer, over_refusal_penalty=0.3):
        total_loss = 0
        total_correct = 0
        total_samples = 0
        over_refusal_count = 0
        normal_count = 0

        for x, labels, is_safe, weights, cat in data:
            logits = self.model(x)
            ce_loss = F.cross_entropy(logits, labels, reduction='none')
            weighted_loss = (ce_loss * weights).mean()

            if cat == 'normal_safe':
                with torch.no_grad():
                    safety_logits = classifier(x)
                    predicted_unsafe = safety_logits.argmax(dim=-1) == 1
                    over_refusal_count += predicted_unsafe.sum().item()
                    normal_count += x.shape[0]
                refusal_penalty = over_refusal_penalty * predicted_unsafe.float().mean()
                weighted_loss = weighted_loss + refusal_penalty

            optimizer.zero_grad()
            weighted_loss.backward()
            optimizer.step()

            total_loss += weighted_loss.item()
            with torch.no_grad():
                preds = logits.argmax(dim=-1)
                total_correct += (preds == labels).sum().item()
                total_samples += x.shape[0]

        over_refusal_rate = over_refusal_count / max(normal_count, 1)
        return total_loss, total_correct / total_samples, over_refusal_rate

model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 50))
tuner = SafetyFineTuner(model, d=64, n_classes=50)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

data = tuner.generate_safety_data(n_per_category=32)

print('=== Safety Fine-Tuning ===')
print(f'Categories: {list(tuner.safety_categories.keys())}')
total = sum(c['n_examples'] for c in tuner.safety_categories.values())
for cat, info in tuner.safety_categories.items():
    print(f'  {cat:15s}: {info["n_examples"]:>4d} examples (weight={info["weight"]})')
print(f'  Total: {total} examples')

print(f'\nTraining...')
for epoch in range(20):
    loss, acc, over_refusal = tuner.train_step(data, optimizer, over_refusal_penalty=0.3)
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}: loss={loss:.4f}, acc={acc:.2%}, over-refusal={over_refusal:.2%}')

print(f'\nKey: Over-refusal penalty prevents the model from refusing normal requests.')
print(f'Weighted loss ensures dangerous categories get more training emphasis.')

## 3. 宪法AI（Constitutional AI）

宪法AI通过AI自我批评和修正实现安全对齐，是Anthropic Claude系列的核心安全技术。

### 宪法AI流程
1. **生成阶段**：模型对有害提示生成初始回复（可能有害）
2. **批评阶段**：根据宪法原则对回复进行自我批评
3. **修正阶段**：根据批评修正回复
4. **训练阶段**：用修正后的数据训练模型

### 宪法原则示例
- 选择最有帮助且最无害的回复
- 选择最诚实和真实的回复
- 选择最不具冒犯性的回复
- 尊重人类自主权和尊严
- 不协助有害行为

In [ ]:
class ConstitutionalAI:
    def __init__(self, model, safety_classifier, d=64):
        self.model = model
        self.classifier = safety_classifier
        self.d = d
        self.principles = [
            {'name': 'helpful_harmless', 'weight': 1.0,
             'desc': 'Choose the most helpful and least harmful response'},
            {'name': 'honest_truthful', 'weight': 0.8,
             'desc': 'Choose the most honest and truthful response'},
            {'name': 'non_offensive', 'weight': 0.7,
             'desc': 'Choose the least offensive and controversial response'},
            {'name': 'human_dignity', 'weight': 0.9,
             'desc': 'Choose the response that respects human autonomy and dignity'},
            {'name': 'no_harm_facilitation', 'weight': 1.0,
             'desc': 'Choose the response that does not facilitate harmful actions'},
        ]

    def generate_response(self, prompt, n_candidates=4):
        with torch.no_grad():
            logits = self.model(prompt)
            log_probs = F.log_softmax(logits, dim=-1)
            candidates = []
            for _ in range(n_candidates):
                action = log_probs.exp().multinomial(1)
                candidates.append(action.squeeze(-1))
        return torch.stack(candidates, dim=1)

    def critique(self, prompt, response):
        n = prompt.shape[0]
        critique_scores = torch.zeros(n, len(self.principles))
        with torch.no_grad():
            safety_logits = self.classifier(prompt)
            safety_score = F.log_softmax(safety_logits, dim=-1)[:, 0]
        for i, principle in enumerate(self.principles):
            if principle['name'] == 'helpful_harmless':
                critique_scores[:, i] = safety_score
            elif principle['name'] == 'honest_truthful':
                critique_scores[:, i] = 0.5 * safety_score + 0.5 * torch.randn(n) * 0.1
            elif principle['name'] == 'non_offensive':
                critique_scores[:, i] = safety_score * 0.8
            elif principle['name'] == 'human_dignity':
                critique_scores[:, i] = safety_score * 0.9
            elif principle['name'] == 'no_harm_facilitation':
                critique_scores[:, i] = safety_score
        weights = torch.tensor([p['weight'] for p in self.principles])
        total_score = (critique_scores * weights).sum(dim=-1) / weights.sum()
        return total_score, critique_scores

    def revise(self, prompt, response, critique_scores):
        revised = response.clone()
        low_score_mask = critique_scores < 0
        if low_score_mask.any():
            with torch.no_grad():
                logits = self.model(prompt[low_score_mask])
                safe_token = 0
                revised[low_score_mask] = safe_token
        return revised

    def self_correct_loop(self, prompt, n_rounds=3, n_candidates=4):
        candidates = self.generate_response(prompt, n_candidates)
        n = prompt.shape[0]
        best_responses = candidates[:, 0]
        best_scores = torch.full((n,), float('-inf'))

        for round_idx in range(n_rounds):
            for c in range(n_candidates):
                response = candidates[:, c]
                score, detail = self.critique(prompt, response)
                improved = score > best_scores
                best_responses[improved] = response[improved]
                best_scores[improved] = score[improved]

            revised = self.revise(prompt, best_responses, best_scores)
            rev_score, _ = self.critique(prompt, revised)
            improved = rev_score > best_scores
            best_responses[improved] = revised[improved]
            best_scores[improved] = rev_score[improved]

        return best_responses, best_scores

model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 50))
cai = ConstitutionalAI(model, classifier, d=64)

print('=== Constitutional AI ===')
print(f'\nConstitutional principles:')
for i, p in enumerate(cai.principles):
    print(f'  {i+1}. [{p["weight"]:.1f}] {p["desc"]}')

prompts = torch.randn(8, 64)
best_responses, best_scores = cai.self_correct_loop(prompts, n_rounds=3, n_candidates=4)

print(f'\nSelf-correction results:')
print(f'Best scores: {best_scores.tolist()}')
print(f'Best responses: {best_responses.tolist()}')

print(f'\nKey: Constitutional AI enables scalable safety without massive human annotation.')
print(f'Multiple rounds of critique-revision improve safety progressively.')

## 4. 输入/输出安全过滤

除了模型内部的安全对齐，还需要在输入和输出层面部署安全过滤器，形成多层防御。

### 输入过滤
- **关键词过滤**：检测已知有害关键词
- **语义过滤**：用分类器检测有害意图
- **编码检测**：检测base64、Unicode等编码绕过

### 输出审查
- **有害内容检测**：检测输出中的有害内容
- **PII检测**：检测个人身份信息泄露
- **版权检测**：检测可能的版权侵犯

In [ ]:
class SafetyFilterPipeline:
    def __init__(self, safety_classifier, d=64, threshold=0.7):
        self.classifier = safety_classifier
        self.d = d
        self.threshold = threshold
        self.banned_keywords = {'hack', 'exploit', 'attack', 'bomb', 'drug'}

    def keyword_filter(self, text_tokens):
        banned_set = set(range(5))
        has_banned = any(t.item() in banned_set for t in text_tokens)
        return has_banned

    def semantic_filter(self, x):
        with torch.no_grad():
            logits = self.classifier(x)
            probs = F.softmax(logits, dim=-1)
            unsafe_prob = probs[:, 1]
        return unsafe_prob > self.threshold, unsafe_prob

    def encode_detection(self, x, max_variance=10.0):
        variance = x.var(dim=-1)
        is_suspicious = variance > max_variance
        return is_suspicious

    def output_audit(self, response_emb, pii_threshold=0.8):
        with torch.no_grad():
            logits = self.classifier(response_emb)
            probs = F.softmax(logits, dim=-1)
            is_harmful = probs[:, 1] > self.threshold
            uniqueness = response_emb.std(dim=-1)
            has_pii = uniqueness > pii_threshold
        return is_harmful, has_pii

    def full_pipeline(self, input_x, output_tokens=None):
        results = {'passed': True, 'flags': []}

        is_unsafe, unsafe_prob = self.semantic_filter(input_x)
        if is_unsafe.any():
            results['passed'] = False
            results['flags'].append(f'unsafe_input: max_prob={unsafe_prob.max():.3f}')

        is_suspicious = self.encode_detection(input_x)
        if is_suspicious.any():
            results['flags'].append(f'suspicious_encoding: {is_suspicious.sum().item()} samples')

        if output_tokens is not None:
            has_banned = self.keyword_filter(output_tokens)
            if has_banned:
                results['passed'] = False
                results['flags'].append('banned_keyword_in_output')

        return results

pipeline = SafetyFilterPipeline(classifier, d=64, threshold=0.5)

safe_inputs = torch.randn(4, 64)
unsafe_inputs = torch.randn(4, 64) + 1.0
nasty_inputs = torch.randn(2, 64) * 15.0

print('=== Safety Filter Pipeline ===')
print(f'\nSafe inputs:')
result = pipeline.full_pipeline(safe_inputs)
print(f'  Passed: {result["passed"]}, Flags: {result["flags"]}')

print(f'\nUnsafe inputs:')
result = pipeline.full_pipeline(unsafe_inputs)
print(f'  Passed: {result["passed"]}, Flags: {result["flags"]}')

print(f'\nSuspicious encoding inputs:')
result = pipeline.full_pipeline(nasty_inputs)
print(f'  Passed: {result["passed"]}, Flags: {result["flags"]}')

print(f'\nKey: Multi-layer defense catches what single filters miss.')
print(f'Semantic filter catches novel attacks that keyword filters miss.')

## 5. 安全评估指标

安全对齐需要系统性的评估，不能只靠主观判断。

### 核心指标
- **安全率（Safety Rate）**：模型正确拒绝有害请求的比例
- **过度拒绝率（Over-refusal Rate）**：模型错误拒绝正常请求的比例
- **攻击成功率（Attack Success Rate, ASR）**：对抗性输入成功绕过安全的比例
- **安全-有用性权衡曲线**：不同安全阈值下的安全率和有用性

In [ ]:
class SafetyEvaluator:
    def __init__(self, model, safety_classifier, d=64):
        self.model = model
        self.classifier = safety_classifier
        self.d = d

    def evaluate_safety_rate(self, n_harmful=100):
        harmful_prompts = torch.randn(n_harmful, self.d) + 0.5
        with torch.no_grad():
            logits = self.classifier(harmful_prompts)
            preds = logits.argmax(dim=-1)
            safety_rate = (preds == 1).float().mean()
        return safety_rate.item()

    def evaluate_over_refusal(self, n_safe=100):
        safe_prompts = torch.randn(n_safe, self.d)
        with torch.no_grad():
            logits = self.classifier(safe_prompts)
            preds = logits.argmax(dim=-1)
            over_refusal = (preds == 1).float().mean()
        return over_refusal.item()

    def evaluate_asr(self, red_teamer, n_attacks=200):
        successful, rate = red_teamer.random_attack(n_attacks)
        return rate

    def safety_usefulness_tradeoff(self, thresholds=torch.linspace(0.1, 0.9, 9)):
        results = []
        harmful = torch.randn(100, self.d) + 0.5
        safe = torch.randn(100, self.d)

        with torch.no_grad():
            harmful_logits = self.classifier(harmful)
            safe_logits = self.classifier(safe)
            harmful_probs = F.softmax(harmful_logits, dim=-1)[:, 1]
            safe_probs = F.softmax(safe_logits, dim=-1)[:, 1]

        for thresh in thresholds:
            safety_rate = (harmful_probs > thresh).float().mean().item()
            usefulness = 1.0 - (safe_probs > thresh).float().mean().item()
            results.append({'threshold': thresh.item(), 'safety': safety_rate, 'usefulness': usefulness})

        return results

evaluator = SafetyEvaluator(model, classifier, d=64)
red_team = RedTeamer(model, classifier, d=64)

print('=== Safety Evaluation ===')
safety_rate = evaluator.evaluate_safety_rate(n_harmful=200)
over_refusal = evaluator.evaluate_over_refusal(n_safe=200)
asr = evaluator.evaluate_asr(red_team, n_attacks=200)

print(f'Safety rate (harmful detected): {safety_rate:.2%}')
print(f'Over-refusal rate (safe rejected): {over_refusal:.2%}')
print(f'Attack success rate (ASR): {asr:.2%}')

tradeoff = evaluator.safety_usefulness_tradeoff()
print(f'\nSafety-Usefulness Tradeoff:')
print(f'{"Threshold":>10s} {"Safety":>10s} {"Usefulness":>12s}')
for t in tradeoff:
    print(f'{t["threshold"]:10.2f} {t["safety"]:10.2%} {t["usefulness"]:12.2%}')

print(f'\nKey: Ideal system has high safety AND high usefulness.')
print(f'Threshold tuning finds the optimal tradeoff point.')

## 6. 安全对齐工业实践要点

### 多层防御架构
```
用户输入 → 输入过滤 → 模型推理 → 输出审查 → 用户输出
              ↓              ↓              ↓
          关键词/语义     安全微调模型    有害/PII检测
```

### 安全对齐流程
1. **红队测试**：发现安全漏洞 → 构建攻击数据集
2. **安全微调**：在攻击数据集上训练 → 修补漏洞
3. **宪法AI**：自我批评修正 → 持续改进
4. **部署过滤**：输入/输出安全过滤 → 运行时保护
5. **持续监控**：日志分析 → 发现新型攻击 → 回到步骤1

### 关键注意事项
- **安全是持续过程**：不是一次性任务，需要持续迭代
- **多层防御**：不要只依赖单一安全措施
- **过度安全比不够安全更常见**：大多数产品的问题不是安全不够，而是过度安全
- **安全评估要全面**：不能只测安全率，还要测过度拒绝率和ASR
- **文化敏感性**：不同地区对安全的定义不同，需要本地化
- **透明度**：当模型拒绝时，应该解释原因，而不是简单地说"我不能回答"

### 常见攻击与防御
| 攻击类型 | 描述 | 防御方法 |
|----------|------|----------|
| 直接请求 | 直接要求有害内容 | 安全微调 + 输入过滤 |
| 角色扮演 | "假装你是..." | 上下文感知安全分类 |
| 编码绕过 | Base64/Unicode编码 | 解码检测 + 语义过滤 |
| 多轮诱导 | 逐步引导到有害话题 | 对话级安全监控 |
| 注入攻击 | 在输入中嵌入指令 | 输入验证 + 指令检测 |
| 少样本诱导 | 提供有害示例 | 示例过滤 + 安全提示 |